In [1]:
import os
import copy
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

In [10]:
# Constants / path
data_dir = "image-results-spoofing/data"
train_dir = os.path.join(data_dir, "Train")
test_dir = os.path.join(data_dir, "Test")
batch_size = 32
learning_rate = 0.001
num_epochs = 200
RANDOM_STATE = 1082
criterion = nn.CrossEntropyLoss()
MODEL_NAME = "CNN"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../UAVATTACK_Results.csv"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
# Data transforms
train_transforms = transforms.Compose([
    transforms.Resize((64, 82)), # Resize to match the input size of photo
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
test_transforms = transforms.Compose([
    transforms.Resize((64, 82)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [5]:
# Datasets and DataLoaders
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

dataloaders = {'train': train_loader, 'test': test_loader}
classes = train_dataset.classes

In [6]:
# CNN Model Definition
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.layer1 = self.ConvModule(in_features=3, out_features=64)  # 16x16
        self.layer2 = self.ConvModule(in_features=64, out_features=128)  # 8x8
        self.layer3 = self.ConvModule(in_features=128, out_features=256)  # 4x4
        self.layer4 = self.ConvModule(in_features=256, out_features=512)  # 2x2
        self.gap = nn.AdaptiveAvgPool2d((2, 2))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * 2 * 512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

    def ConvModule(self, in_features, out_features):
        return nn.Sequential(
            nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_features),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )


model = CNN(num_classes=len(classes)).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [7]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs, device):
    since = time.time()
    history = {'train_loss': [], 'train_accuracy': [], 'test_loss': [], 'test_accuracy': []}

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += preds.eq(labels).sum().item()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects / len(dataloaders[phase].dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2%}')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_accuracy'].append(epoch_acc)

            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best test Acc: {best_acc:.2%}')

    model.load_state_dict(best_model_wts)
    return model, history

# Train
model, history = train_model(model, dataloaders, criterion, optimizer, num_epochs, device)


Epoch 1/200
Train Loss: 0.4173, Acc: 85.03%
Test Loss: 0.3975, Acc: 86.31%

Epoch 2/200
Train Loss: 0.0028, Acc: 99.93%
Test Loss: 0.0094, Acc: 99.59%

Epoch 135/200
Train Loss: 0.0006, Acc: 100.00%
Test Loss: 0.0071, Acc: 99.59%

Epoch 136/200
Train Loss: 0.0013, Acc: 99.93%
Test Loss: 0.0914, Acc: 98.34%

Epoch 137/200
Train Loss: 0.0013, Acc: 99.97%
Test Loss: 0.0048, Acc: 99.59%

Epoch 138/200
Train Loss: 0.0009, Acc: 99.97%
Test Loss: 0.0125, Acc: 99.59%

Epoch 139/200
Train Loss: 0.0005, Acc: 100.00%
Test Loss: 0.0110, Acc: 99.59%

Epoch 140/200
Train Loss: 0.0003, Acc: 100.00%
Test Loss: 0.0053, Acc: 99.59%

Epoch 141/200
Train Loss: 0.0004, Acc: 100.00%
Test Loss: 0.0074, Acc: 99.72%

Epoch 142/200
Train Loss: 0.0003, Acc: 100.00%
Test Loss: 0.0061, Acc: 99.72%

Epoch 143/200
Train Loss: 0.0016, Acc: 99.97%
Test Loss: 0.0027, Acc: 99.86%

Epoch 144/200
Train Loss: 0.0002, Acc: 100.00%
Test Loss: 0.0075, Acc: 99.45%

Epoch 145/200
Train Loss: 0.0004, Acc: 100.00%
Test Loss: 0.01

In [8]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

Test Accuracy: 99.86%
Test F1 Score: 0.9986


In [11]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy
}]
results_df = pd.DataFrame(results)
file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../UAVATTACK_Results.csv
